# Parte 4 — PINN: Estimación de parámetros Saint-Venant 1D

**Physics-Informed Neural Network** que aprende A(x,t) y Q(x,t) como funciones
continuas y estima los parámetros físicos n y B_w minimizando:

- **L_data** = MSE entre Q predicho en x=0 y x=L vs observaciones del CSV
  *(ancla la solución a los datos reales en los bordes)*
- **L_pde** = residuos de las ecuaciones de Saint-Venant calculados por `torch.autograd`
  en 2 000 puntos de colocación interiores
  *(hace que la red respete la física en todo el dominio, no solo en los bordes)*

**Por qué se necesitan los dos términos:** con solo L_data, el problema es no
identificable — infinitas combinaciones de (n, B_w, A, Q) ajustan los bordes.
L_pde resuelve la identifiabilidad porque n aparece en Sf y B_w en la geometría
hidráulica: ambos deben hacer que los residuos SV sean ≈ 0 en el interior.

**Entrenamiento:** Fase 1 Adam (exploración amplia) → Fase 2 L-BFGS (convergencia fina).


In [ ]:
# ── Configuración ─────────────────────────────────────────────────────────────
# Cambiar True/False para estimar otros parámetros
ESTIMATE_PARAMS = {"n": True, "Bw": True}

# Pesos de la función de pérdida
LAMBDA_DATA = 1.0    # peso de L_data (ajuste a observaciones)
LAMBDA_PDE  = 0.05   # peso de L_pde  (cumplimiento de la física)

# Hiperparámetros de entrenamiento
N_EPOCHS_ADAM    = 6_000   # total épocas Fase 1 — Adam
N_EPOCHS_WARMUP  = 2_000   # primeras N épocas sin física (solo datos)
N_ITER_LBFGS     = 500     # iteraciones Fase 2 — L-BFGS
N_COLLOC         = 2_000   # puntos de colocación interiores
RESAMPLE_EVERY   = 1_000   # re-muestrear colloc cada N épocas Adam

# Conjeturas iniciales (lejos del valor verdadero para mostrar convergencia)
N_INIT  = 0.030   # verdadero: 0.035
BW_INIT = 45.0    # verdadero: 50.0 m

# Archivo de observaciones (mismo que en calibración, notebook 03)
CSV_NAME    = "series_corta_shift.csv"
WARMUP_SEC  = 3600.0


In [ ]:
# ── Carga de datos ─────────────────────────────────────────────────────────────
import importlib, importlib.util, sys
from pathlib import Path
import numpy as np, pandas as pd, torch

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# Parámetros verdaderos desde sinteticos.py
_spec = importlib.util.spec_from_file_location("sinteticos", ROOT / "data" / "sinteticos.py")
sinteticos = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(sinteticos)
TRUE_N, TRUE_BW = sinteticos.N_MANN, sinteticos.B_W

csv_path = ROOT / "data" / "synthetic" / CSV_NAME
df   = pd.read_csv(csv_path, parse_dates=["datetime"])
t_sec = (df["datetime"] - df["datetime"].iloc[0]).dt.total_seconds().to_numpy(dtype=float)
q_up  = df["Q_upstream_m3s"].to_numpy(dtype=float)
q_dn  = df["Q_downstream_m3s"].to_numpy(dtype=float)

t_tensor  = torch.tensor(t_sec, dtype=torch.float32)
qu_tensor = torch.tensor(q_up,  dtype=torch.float32)
qd_tensor = torch.tensor(q_dn,  dtype=torch.float32)
T_total   = float(t_sec[-1])

print(f"CSV: {CSV_NAME}  |  pasos: {len(df)}  |  T={T_total/3600:.1f} h")
print(f"Valores verdaderos: n={TRUE_N}  B_w={TRUE_BW} m")


In [ ]:
# ── Construcción y entrenamiento ──────────────────────────────────────────────
import importlib
import src.pinn as pinn_mod
importlib.reload(pinn_mod)

model = pinn_mod.SVPINN(
    hidden_size=64,
    n_layers=4,
    S0=sinteticos.S0,
    beta=1.0,
    n_init=N_INIT,
    Bw_init=BW_INIT,
    estimate_params=ESTIMATE_PARAMS,
)

# Curriculum: Fase 1a (solo datos, N_EPOCHS_WARMUP épocas) →
#             Fase 1b (datos + física, hasta N_EPOCHS_ADAM) →
#             Fase 2  (L-BFGS convergencia fina)
result = pinn_mod.train(
    model,
    x0_data=qu_tensor,
    xL_data=qd_tensor,
    t_data=t_tensor,
    L=sinteticos.L,
    T=T_total,
    lambda_data=LAMBDA_DATA,
    lambda_pde=LAMBDA_PDE,
    n_epochs_adam=N_EPOCHS_ADAM,
    n_epochs_warmup=N_EPOCHS_WARMUP,
    n_iter_lbfgs=N_ITER_LBFGS,
    n_colloc=N_COLLOC,
    resample_every=RESAMPLE_EVERY,
    t_warmup=WARMUP_SEC,
    verbose_every=500,
)
print(f"\nEstimado PINN: n={result.n_estimate:.5f}  Bw={result.Bw_estimate:.3f} m")


In [ ]:
# ── Curvas de pérdida + estimación de parámetros ──────────────────────────────
import matplotlib.pyplot as plt
from IPython.display import display

history = result.loss_history
adam_entries = [h for h in history if "data" in h]
epochs_a = [h["epoch"] for h in adam_entries]
totals_a = [h["total"] for h in adam_entries]
datas_a  = [h["data"]  for h in adam_entries]
pdes_a   = [h["pde"]   for h in adam_entries]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Curvas de pérdida (fase Adam, eje log)
ax = axes[0]
ax.semilogy(epochs_a, totals_a, label="L_total", color="k")
ax.semilogy(epochs_a, datas_a,  label="L_data",  color="steelblue")
ax.semilogy(epochs_a, pdes_a,   label=f"L_pde (x{LAMBDA_PDE})", color="coral")
ax.set_xlabel("Época (Adam)")
ax.set_ylabel("Pérdida (escala log)")
ax.set_title("Curvas de pérdida — fase Adam")
ax.legend(); ax.grid(True, alpha=0.3)

# Tabla y gráfico de parámetros
glue_path = ROOT / "data" / "synthetic" / "glue_parametros_aceptados.csv"
table = pinn_mod.build_comparison_table(
    n_pinn=result.n_estimate, Bw_pinn=result.Bw_estimate,
    n_true=TRUE_N, Bw_true=TRUE_BW,
    glue_csv_path=str(glue_path),
)
display(table)

params     = ["n (Manning)", "B_w [m]"]
verdaderos = [TRUE_N, TRUE_BW]
pinns      = [result.n_estimate, result.Bw_estimate]
use_glue   = "calibracion_GLUE" in table.columns
calibs     = list(table["calibracion_GLUE"]) if use_glue else [np.nan, np.nan]

ax2 = axes[1]
x = np.arange(len(params)); w = 0.25
ax2.bar(x - w, verdaderos, w, label="Verdadero",         color="0.70")
ax2.bar(x,     pinns,      w, label="PINN",               color="steelblue")
if use_glue and not any(np.isnan(np.array(calibs, dtype=float))):
    ax2.bar(x + w, calibs, w, label="Calibración (GLUE)", color="coral")
ax2.set_xticks(x); ax2.set_xticklabels(params)
ax2.set_title("Parámetros estimados")
ax2.legend(); ax2.grid(True, axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(ROOT / "figures" / "pinn_parametros.png", dpi=150)
plt.show()


In [ ]:
# ── Hidrograma: PINN vs observado en x=L ─────────────────────────────────────
model.eval()
with torch.no_grad():
    t_norm_all = t_tensor / T_total
    _, Q_pinn  = model(torch.ones_like(t_norm_all), t_norm_all)
Q_pinn_np = Q_pinn.numpy()

t_h       = t_sec / 3600.0
mask_post = t_sec >= WARMUP_SEC

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t_h[mask_post], q_dn[mask_post],    "r.",        ms=3,  alpha=0.6, label="Q_obs  (x=L)")
ax.plot(t_h[mask_post], Q_pinn_np[mask_post], "steelblue", lw=1.5, label="Q_PINN (x=L)")
ax.axvspan(0, WARMUP_SEC / 3600, color="gray", alpha=0.12, label="Warm-up")
ax.set_xlabel("Tiempo (h)")
ax.set_ylabel("Q (m³/s)")
ax.set_title(
    f"Hidrograma salida — PINN vs observado\n"
    f"n={result.n_estimate:.4f} (verd. {TRUE_N})  "
    f"Bw={result.Bw_estimate:.2f} m (verd. {TRUE_BW} m)"
)
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(ROOT / "figures" / "pinn_hidrograma.png", dpi=150)
plt.show()
